# Citation-Aware Hybrid Legal Case Retrieval for Indian Court Judgments
### Information Retrieval Course Project | Author: Vismaya
---
This notebook runs the complete end-to-end Legal Precedent Retrieval pipeline on **Google Colab (Free T4 GPU)**.

**Pipeline Architecture:**
1. **Sparse Lexical Retrieval**: BM25+ with Indian legal stopword filtering and statutory token retention.
2. **Dense Semantic Retrieval**: Chunk-pooled embeddings using `law-ai/InLegalBERT` or `sentence-transformers/all-MiniLM-L6-v2`.
3. **Citation Network Graph**: Directed citation network computing In-Degree Centrality, PageRank, Personalized PageRank (PPR), and Bibliographic Coupling.
4. **Hybrid Reranking**: Supervised LambdaMART (XGBoost) combining lexical, dense, graph, and statutory overlap features.
5. **Academic Evaluation & Ablation Suite**: nDCG@5, nDCG@10, MAP, Recall@{5,10,20}, and MRR.

## 1. Environment & Hardware Verification

In [ ]:
!nvidia-smi
!python --version

## 2. Install Project Dependencies

In [ ]:
!pip install --quiet rank-bm25 sentence-transformers xgboost networkx tabulate pydantic pyyaml rich pytest

## 3. Verify Deterministic Reproducibility & Unit Tests

In [ ]:
!pytest tests/ -v

## 4. Run Individual Pipeline Phases
You can test individual phases independently using `--mode`:

In [ ]:
# Phase 1: Data Ingestion & Preprocessing
!python run_pipeline.py --mode data --dataset sample

In [ ]:
# Phase 2: Sparse BM25 Retrieval Baseline
!python run_pipeline.py --mode sparse --dataset sample

In [ ]:
# Phase 4: Citation Graph Construction & Authority Metrics
!python run_pipeline.py --mode graph --dataset sample

In [ ]:
# Phase 3: Dense Semantic Retrieval on GPU
!python run_pipeline.py --mode dense --dataset sample

## 5. Execute Full Multi-Stage Ablation Experiment
Runs all 5 comparative models:
1. BM25 Alone
2. Dense Alone
3. Hybrid (BM25 + Dense RRF)
4. Hybrid + Citation Graph (PPR)
5. Full LambdaMART Reranker

In [ ]:
!python run_pipeline.py --mode all --config configs/colab_config.yaml

## 6. Inspect Ablation Results & Feature Importance

In [ ]:
import pandas as pd
import json

with open('experiments/results/ablation_results.json', 'r') as f:
    results = json.load(f)

df = pd.DataFrame(results)
display(df)

# Display Markdown Summary
with open('experiments/results/ablation_summary.md', 'r') as f:
    print(f.read())